# BirdCLEF+ 2026 — Phase 3: Perch v2 Embedding Extraction

This notebook precomputes **Perch v2 embeddings** for every training example we'll use:

1. **Clips**: center 5-second slice of each clip in `train_audio/` — 1 embedding per clip × ~35,549 clips.
2. **Labeled soundscape windows**: the exact (file, start, end) windows in `train_soundscapes_labels.csv` — 1 embedding per row × 1,478 rows.

Each embedding is a 1536-d float32 vector — a "fingerprint" that Perch (pretrained on millions of bird recordings) produced for the audio.

Output files (saved to `embeddings/`):
- `clip_embeddings.npy` — shape `(N_clips, 1536)` float32
- `clip_index.csv` — columns: `filename, primary_label, secondary_labels`
- `ss_window_embeddings.npy` — shape `(1478, 1536)` float32
- `ss_window_index.csv` — columns: `filename, start_sec, end_sec, primary_label`

**Expected runtime**: ~30-60 min total on Apple Silicon CPU. Run once; the cached embeddings power Phase 3+ training and inference. Resumable — if interrupted, just rerun.


## 1. Setup


In [1]:
import os, time
from pathlib import Path

import numpy as np
import pandas as pd
import librosa
import soundfile as sf
import onnxruntime as ort
from tqdm.auto import tqdm

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR     = PROJECT_ROOT / "data" / "birdclef-2026"
TRAIN_AUDIO  = DATA_DIR / "train_audio"
SS_AUDIO_DIR = DATA_DIR / "train_soundscapes"
PERCH_PATH   = PROJECT_ROOT / "data" / "perch" / "perch_v2_no_dft.onnx"
EMBED_DIR    = PROJECT_ROOT / "embeddings"
EMBED_DIR.mkdir(exist_ok=True)

assert PERCH_PATH.exists(), f"Missing {PERCH_PATH} — run kaggle datasets download first"

SR        = 32000
CLIP_SEC  = 5
N_SAMPLES = SR * CLIP_SEC  # 160000
EMBED_DIM = 1536

print(f"Perch ONNX:  {PERCH_PATH}  ({PERCH_PATH.stat().st_size/1e6:.0f} MB)")
print(f"Embed dir:   {EMBED_DIR}")


Perch ONNX:  /Users/harish.r/Documents/kaggle/birdclef2026/data/perch/perch_v2_no_dft.onnx  (413 MB)
Embed dir:   /Users/harish.r/Documents/kaggle/birdclef2026/embeddings


## 2. Load Perch + sanity check


In [2]:
# CPU is fastest for this model on macOS arm64 (CoreML is buggy at higher batches and slower at batch=1).
sess = ort.InferenceSession(str(PERCH_PATH), providers=["CPUExecutionProvider"])
INPUT_NAME    = sess.get_inputs()[0].name           # "inputs"
EMBED_OUT_IDX = next(i for i, o in enumerate(sess.get_outputs()) if o.name == "embedding")

# Smoke-test with a dummy waveform
_dummy = np.random.randn(1, N_SAMPLES).astype(np.float32)
_out   = sess.run(None, {INPUT_NAME: _dummy})
print(f"Perch ready. Embedding shape: {_out[EMBED_OUT_IDX].shape}  ({_out[EMBED_OUT_IDX].dtype})")


Perch ready. Embedding shape: (1, 1536)  (float32)


## 3. Audio helpers

`load_audio` reads any .ogg file as float32 mono at 32 kHz. `take_center_5s`
deterministically takes the center 5 seconds (pads if shorter).


In [3]:
def load_audio(path):
    wav, sr = sf.read(str(path), dtype="float32", always_2d=False)
    if wav.ndim > 1:
        wav = wav.mean(axis=1)
    if sr != SR:
        wav = librosa.resample(wav, orig_sr=sr, target_sr=SR)
    return wav.astype(np.float32)


def take_center_5s(wav):
    if len(wav) < N_SAMPLES:
        # Pad symmetrically around the center
        pad = N_SAMPLES - len(wav)
        left = pad // 2
        right = pad - left
        wav = np.pad(wav, (left, right))
    elif len(wav) > N_SAMPLES:
        start = (len(wav) - N_SAMPLES) // 2
        wav = wav[start:start + N_SAMPLES]
    return wav.astype(np.float32)


def slice_window(wav, start_sec, end_sec):
    """Slice an exact window (in samples) from a waveform, padding if short."""
    start_samp = start_sec * SR
    end_samp   = end_sec * SR
    if len(wav) < end_samp:
        wav = np.pad(wav, (0, end_samp - len(wav)))
    chunk = wav[start_samp:end_samp]
    if len(chunk) < N_SAMPLES:
        chunk = np.pad(chunk, (0, N_SAMPLES - len(chunk)))
    elif len(chunk) > N_SAMPLES:
        chunk = chunk[:N_SAMPLES]
    return chunk.astype(np.float32)


## 4. Extract embeddings — clips

For each row in `train.csv`, load the audio, take the center 5 seconds, run
Perch → save the 1536-d embedding.

We save incrementally every 1,000 clips so an interruption only loses a small
batch of work. On rerun we resume from the last saved checkpoint.


In [4]:
df = pd.read_csv(DATA_DIR / "train.csv")
df["primary_label"] = df["primary_label"].astype(str)
print(f"Total clips: {len(df):,}")

# Output paths
clip_emb_path   = EMBED_DIR / "clip_embeddings.npy"
clip_index_path = EMBED_DIR / "clip_index.csv"
clip_progress   = EMBED_DIR / "clip_progress.txt"

# Resume support: load existing embeddings if present
if clip_emb_path.exists() and clip_progress.exists():
    embeddings = np.load(clip_emb_path)
    start_idx  = int(clip_progress.read_text().strip())
    print(f"Resuming from index {start_idx} ({start_idx/len(df)*100:.1f}% done)")
else:
    embeddings = np.zeros((len(df), EMBED_DIM), dtype=np.float32)
    start_idx  = 0

# Save the index CSV up front (so it matches embedding rows by position)
df[["filename", "primary_label", "secondary_labels"]].to_csv(clip_index_path, index=False)

BATCH_SIZE = 8
SAVE_EVERY = 1000

t0 = time.time()
i = start_idx
pbar = tqdm(total=len(df), initial=start_idx, desc="clips")
while i < len(df):
    # Build a batch of waveforms
    batch_idx = list(range(i, min(i + BATCH_SIZE, len(df))))
    batch_wavs = []
    for j in batch_idx:
        path = TRAIN_AUDIO / df.iloc[j]["filename"]
        try:
            wav = load_audio(path)
            wav = take_center_5s(wav)
        except Exception as e:
            print(f"[skip] {path}: {e}")
            wav = np.zeros(N_SAMPLES, dtype=np.float32)
        batch_wavs.append(wav)

    batch_input = np.stack(batch_wavs)
    outs = sess.run(None, {INPUT_NAME: batch_input})
    embeddings[i:i+len(batch_idx)] = outs[EMBED_OUT_IDX]

    i += len(batch_idx)
    pbar.update(len(batch_idx))

    if i % SAVE_EVERY < BATCH_SIZE:
        np.save(clip_emb_path, embeddings)
        clip_progress.write_text(str(i))
pbar.close()

# Final save
np.save(clip_emb_path, embeddings)
clip_progress.write_text(str(i))
dt = time.time() - t0
print(f"\nDone. {i:,} clips in {dt/60:.1f} min  ({i/max(dt,1e-9):.1f} wins/s)")
print(f"Saved: {clip_emb_path}  shape={embeddings.shape}  size={clip_emb_path.stat().st_size/1e6:.1f} MB")


Total clips: 35,549


clips:   0%|          | 0/35549 [00:00<?, ?it/s]


Done. 35,549 clips in 27.5 min  (21.6 wins/s)
Saved: /Users/harish.r/Documents/kaggle/birdclef2026/embeddings/clip_embeddings.npy  shape=(35549, 1536)  size=218.4 MB


## 5. Extract embeddings — labeled soundscape windows

For each row in `train_soundscapes_labels.csv`, load the soundscape, slice the
exact 5s window, run Perch. These windows have multi-species labels (semicolon-
separated).

Important optimization: each soundscape file appears in ~22 rows on average
(12-22 labeled windows per file). We **load each file once** and reuse the
waveform across all its rows.


In [5]:
def parse_time_to_sec(s):
    h, m, ss = s.split(":")
    return int(h) * 3600 + int(m) * 60 + int(ss)

ss_df = pd.read_csv(DATA_DIR / "train_soundscapes_labels.csv")
ss_df["start_sec"] = ss_df["start"].apply(parse_time_to_sec)
ss_df["end_sec"]   = ss_df["end"].apply(parse_time_to_sec)
print(f"Labeled soundscape windows: {len(ss_df):,}")
print(f"Unique files: {ss_df['filename'].nunique()}")

ss_emb_path   = EMBED_DIR / "ss_window_embeddings.npy"
ss_index_path = EMBED_DIR / "ss_window_index.csv"
ss_progress   = EMBED_DIR / "ss_progress.txt"

if ss_emb_path.exists() and ss_progress.exists():
    ss_embeddings = np.load(ss_emb_path)
    start_idx     = int(ss_progress.read_text().strip())
    print(f"Resuming from index {start_idx}")
else:
    ss_embeddings = np.zeros((len(ss_df), EMBED_DIM), dtype=np.float32)
    start_idx     = 0

ss_df[["filename", "start_sec", "end_sec", "primary_label"]].to_csv(ss_index_path, index=False)

t0 = time.time()
i = start_idx
pbar = tqdm(total=len(ss_df), initial=start_idx, desc="soundscape wins")
current_file = None
current_wav  = None

while i < len(ss_df):
    # Build a batch — but only group rows from the same file (so we cache the file load)
    batch_idx = []
    batch_wavs = []
    while len(batch_idx) < BATCH_SIZE and i + len(batch_idx) < len(ss_df):
        row = ss_df.iloc[i + len(batch_idx)]
        # Load file if it changed
        if row["filename"] != current_file:
            try:
                current_wav  = load_audio(SS_AUDIO_DIR / row["filename"])
                current_file = row["filename"]
            except Exception as e:
                print(f"[skip] {row['filename']}: {e}")
                current_wav = np.zeros(60 * SR, dtype=np.float32)
        chunk = slice_window(current_wav, row["start_sec"], row["end_sec"])
        batch_idx.append(i + len(batch_idx))
        batch_wavs.append(chunk)

    if not batch_wavs:
        break

    batch_input = np.stack(batch_wavs)
    outs = sess.run(None, {INPUT_NAME: batch_input})
    ss_embeddings[batch_idx[0]:batch_idx[0]+len(batch_wavs)] = outs[EMBED_OUT_IDX]

    i = batch_idx[-1] + 1
    pbar.update(len(batch_wavs))

    if i % 200 < BATCH_SIZE:
        np.save(ss_emb_path, ss_embeddings)
        ss_progress.write_text(str(i))
pbar.close()

np.save(ss_emb_path, ss_embeddings)
ss_progress.write_text(str(i))
dt = time.time() - t0
print(f"\nDone. {i:,} windows in {dt/60:.1f} min")
print(f"Saved: {ss_emb_path}  shape={ss_embeddings.shape}  size={ss_emb_path.stat().st_size/1e6:.1f} MB")


Labeled soundscape windows: 1,478
Unique files: 66


soundscape wins:   0%|          | 0/1478 [00:00<?, ?it/s]


Done. 1,478 windows in 0.8 min
Saved: /Users/harish.r/Documents/kaggle/birdclef2026/embeddings/ss_window_embeddings.npy  shape=(1478, 1536)  size=9.1 MB


## 6. Sanity check the embeddings

Three quick checks:

1. Embeddings should be non-trivial (not all zeros). Spot-check a few.
2. Same-species clips should have somewhat similar embeddings (cosine similarity > random).
3. Different-species clips should have lower similarity than same-species.

If these don't hold roughly, something's broken in the extraction.


In [6]:
clip_idx_df = pd.read_csv(clip_index_path)
clip_emb    = np.load(clip_emb_path)

# Non-zero check
nonzero = (np.abs(clip_emb).sum(axis=1) > 0)
print(f"Non-zero embeddings: {nonzero.sum():,} / {len(clip_emb):,}")

# Quick cosine similarity test
def cos_sim(a, b):
    return (a @ b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-9)

# Pick a species with many clips
species_counts = clip_idx_df["primary_label"].value_counts()
target = species_counts.head(1).index[0]
same_idx = clip_idx_df[clip_idx_df["primary_label"] == target].index[:10].tolist()
diff_idx = clip_idx_df[clip_idx_df["primary_label"] != target].sample(10, random_state=0).index.tolist()

same_sims = [cos_sim(clip_emb[same_idx[0]], clip_emb[j]) for j in same_idx[1:]]
diff_sims = [cos_sim(clip_emb[same_idx[0]], clip_emb[j]) for j in diff_idx]

print(f"\nSpecies {target}:")
print(f"  Same-species cosine sim (n={len(same_sims)}): mean={np.mean(same_sims):.3f}  range=[{min(same_sims):.3f}, {max(same_sims):.3f}]")
print(f"  Diff-species cosine sim (n={len(diff_sims)}): mean={np.mean(diff_sims):.3f}  range=[{min(diff_sims):.3f}, {max(diff_sims):.3f}]")
print(f"\nIf same-species mean > diff-species mean, embeddings are encoding species patterns. ✓")


Non-zero embeddings: 35,549 / 35,549

Species rubthr1:
  Same-species cosine sim (n=9): mean=0.255  range=[0.014, 0.399]
  Diff-species cosine sim (n=10): mean=0.112  range=[-0.021, 0.301]

If same-species mean > diff-species mean, embeddings are encoding species patterns. ✓


## What's next

When this finishes, embeddings live at `embeddings/`:

- `clip_embeddings.npy` (~210 MB)
- `clip_index.csv`
- `ss_window_embeddings.npy` (~9 MB)
- `ss_window_index.csv`

These power Phase 3 head training (`01_train.ipynb` rewrite — coming next).
The next notebook trains a small classifier on these cached embeddings — fast
(~minutes per epoch instead of hours), so we can iterate rapidly.
